In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

data=pd.read_csv("pos_tags.csv",nrows=1000) #1000 rows of datas only got fetched out of lakhs
print(data.head())

sentences=[]
temp=[]
for _,row in data.iterrows():
    word=row["word"]
    tag=row["tag"]
    temp.append((word,tag))
    if(len(temp)==20):
        sentences.append(temp)
        temp=[]
print(sentences)
train_data,test_data=train_test_split(sentences,test_size=0.2,random_state=42)
#create vocabulary - stores unique terms
vocabulary=set()
tags=set()
for sentence in train_data:
    for word,tag in sentence:
        vocabulary.add(word.lower())
        tags.add(tag)
print(vocabulary)
print(tags)
vocabulary=list(vocabulary) #convert the set of vocabulary back to set
tags=list(tags)
word_to_index={word:i for i,word in enumerate(vocabulary)} #converting to numbers coz it can't understand the text
tag_to_index={tag:i for i,tag in enumerate(tags)}
index_to_tag={i:tag for tag,i in tag_to_index.items()}
V=len(vocabulary)
T=len(tags)
print("\nVocabulary Size:",V)
print("Number of POS Tags:",T)

#HMM Matrices
#Initial,transition,emission probability
#initial - probability(starting tag)
#transition - relationship between two pos tags -> P(current tag|previous tag)
#emission - relation between word and tag -> P(word|tag) (works diagonally)

initial=np.ones(T) #np.ones fills the matrix with 1
transition=np.ones((T,T))
emission=np.ones((T,V))

#count how many times each tag appears
for sentence in train_data:
    first_tag=sentence[0][1]
    initial[tag_to_index[first_tag]]+=1

    for i,(word,tag) in enumerate(sentence):
        word=word.lower()
        tag_index=tag_to_index[tag]

        #emission count
        if word in word_to_index:
            emission[tag_index,word_to_index[word]]+=1 

        #transition count
        if i>0:
            previous_tag=sentence[i-1][1]
            transition[tag_to_index[previous_tag],tag_index]+=1

#convert the count to probability value
initial/=initial.sum()
transition/=transition.sum(axis=1,keepdims=True) #axis=1 means calculate row wise axis=0 means calculate column wise
emission/=emission.sum(axis=1,keepdims=True) #keepdims -> dimensions

#convert to log
log_initial=np.log(initial)
log_transition=np.log(transition)
log_emission=np.log(emission)

#viterbi algorithm

def viterbi(sentence):
    n=len(sentence)
    dp=np.zeros((T,n)) #creates a matrix with 0 based on len of sentence and len of tag(count)
    backpointer=np.zeros((T,n),dtype=int) #stores previous tag 
    #first word
    word=sentence[0].lower()
    if word in word_to_index:
        emit=log_emission[:,word_to_index[word]]
    else:
        emit=np.log(np.ones(T)*1e-10)
    dp[:,0]=(log_initial+emit)

    #remaining words
    for i in range(1,n):
        word=sentence[i].lower()
        if word in word_to_index:
            emit=log_emission[:,word_to_index[word]]
        else:
            emit=np.log(np.ones(T)*1e-10)
    
        #vector calculation
        scores=(dp[:,i-1][:,None]+log_transition)
        backpointer[:,i]=np.argmax(scores,axis=0)
        dp[:,i]=(np.max(scores,axis=0)+emit)

    #backtracking
    best=np.argmax(dp[:,-1])
    result=[best]
    for i in range(n-1,0,-1):
        best=backpointer[best,i]
        result.append(best)
    result.reverse()
    return[index_to_tag[i] for i in result]

sentence="Artificial intelligence improves healthcare systems".split()
prediction=viterbi(sentence)
print("\nPrediction")
print("----------------")
for word,tag in zip(sentence,prediction):
    print(word,"-->",tag)

#Evaluation
actual=[]
predicted=[]

for sentence in test_data:
    words=[w for w,t in sentence]
    true=[t for w,t in sentence]
    pred=viterbi(words)
    actual.extend(true)
    predicted.extend(pred)
print("\nAccuracy")
print(accuracy_score(actual,predicted))
print("\nClassification Report")
print(classification_report(actual,predicted,zero_division=0))





   sentence_id    word  tag
0            0      aa   NN
1            1     aaa   NN
2            2     aah   NN
3            3   aahed  VBN
4            4  aahing  VBG
[[('aa', 'NN'), ('aaa', 'NN'), ('aah', 'NN'), ('aahed', 'VBN'), ('aahing', 'VBG'), ('aahs', 'NN'), ('aal', 'NN'), ('aalii', 'NN'), ('aaliis', 'NN'), ('aals', 'NNS'), ('aam', 'NN'), ('aani', 'NN'), ('aardvark', 'NN'), ('aardvarks', 'NNS'), ('aardwolf', 'NN'), ('aardwolves', 'NNS'), ('aargh', 'NN'), ('aaron', 'NN'), ('aaronic', 'NN'), ('aaronical', 'JJ')], [('aaronite', 'NN'), ('aaronitic', 'JJ'), ('aarrgh', 'NN'), ('aarrghh', 'NN'), ('aaru', 'NN'), ('aas', 'NN'), ('aasvogel', 'NN'), ('aasvogels', 'NNS'), ('ab', 'NN'), ('aba', 'NN'), ('ababdeh', 'NN'), ('ababua', 'NN'), ('abac', 'NN'), ('abaca', 'NN'), ('abacay', 'NN'), ('abacas', 'NN'), ('abacate', 'NN'), ('abacaxi', 'NN'), ('abaci', 'NN'), ('abacinate', 'NN')], [('abacination', 'NN'), ('abacisci', 'NN'), ('abaciscus', 'NN'), ('abacist', 'NN'), ('aback', 'NN'), ('abacli',